# Stream NISAR GCOV Data with `asf_search`, `s3fs`, and `xarray`

<br>

[`asf_search`](https://github.com/asfadmin/Discovery-asf_search) is an open source Python package for searching SAR data archived at {abbr}`ASF (Alaska Satellite Facility)`. This notebook demonstrates how to search NISAR GCOV with `asf_search`, stream them directly from S3 storage with `s3fs`, and load them into `xarray` data structures.

NISAR data products can be very large. It may be helpful to access the data directly from their S3 storage. This allows you to lazily load data into `xarray` data structures, subset, and perform operations on them with `xarray`, and then only save the data you need to memory. This avoids downloading many large products to a storage volume, only to subset and delete most of them. The caveat is that you must have enough RAM to hold your final subset and also run your workflow.

:::{warning} You can only access NISAR data directly from an S3 bucket if you are working in the `us-west-2` AWS region

Often, this means that you are working on an AWS EC2 instance in the `us-west-2` region, though you could possibly be working with another in-region AWS service, such as Lambda, Fargate, SageMaker, etc...

You could also be working on an in-region, AWS-hosted JupyterHub service such as ASF OpenScienceLab.

**Direct S3 access is not possible from a Python script or Jupyter Notebook running directly on your computer or HPC.**

:::

<hr>

## Overview

1. [Prerequisites](asf_search_stream-prereqs)
1. [Search for data](asf_search_stream-search)
1. [(Option 1) Load a single GCOV product using a Python `with` statement](asf_search_stream-with)
1. [(Option 2) Load multiple GCOV Products at once](asf_search_stream-open_many)
1. [Stream a time series](asf_search_stream-ts)
1. [Close open files](asf_search_stream-close)
1. [Summary](asf_search_stream-summary)
1. [Resources and references](asf_search_stream-resources)

<hr>

(asf_search_stream-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(asf_search_stream-search)=
## 3. Search for data

Use `asf_search` to find GCOV data.

### 3a. Perform an `asf_search.search()` to identify your desired product URLs


In [1]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass

import warnings
warnings.filterwarnings(
    "ignore",
    message="Parsing dates involving a day of month without a year specified",
)

session = asf.ASFSession()

start_date = datetime(2025, 11, 22)
end_date = datetime(2026, 3, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" # POINT or POLYGON as WKT (well-known-text)
pattern = r'^(?!.*QA_STATS).*'

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "flightDirection": "ASCENDING",
    "start": start_date,
    "end": end_date,
    "processingLevel": [
        "GCOV"
    ],
    "dataset": [
        "NISAR"
    ],
    "productionConfiguration": [
        "PR"
    ],
    'session': session,
})

response = asf.search(opts=opts)
hdf5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
print(f"Found {len(hdf5_urls)} GCOV products:")
hdf5_urls

Found 7 GCOV products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T0

### 3b. Retain only the URL for the most recent version of each product in the search results

Data is occasionally re-released with an updated version. Versions are recorded as a [Composite Release Identifier (CRID)](https://nisar-docs.asf.alaska.edu/naming-conventions/) in a product's filename. We can use the CRID to retain only the most recent version of each product in the list of URLs.

In [2]:
import re

pattern = re.compile(r"(NISAR_L2_PR_GCOV(?:_[^_]+){9})_(X\d{5})")

latest_version_dict = {}

for url in hdf5_urls:
    m = pattern.search(url)
    if not m:
        continue

    product, crid = m.groups()

    if product not in latest_version_dict or crid > latest_version_dict[product][0]:
        latest_version_dict[product] = (crid, url)

hdf5_urls = [i[1] for i in latest_version_dict.values()]
print(f"Retained {len(hdf5_urls)} GCOV products:")
hdf5_urls

Retained 5 GCOV products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T024619_20251228T024654_X05009_N_F_J_001/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T0

### 3c. Convert the hdf5 URLs into S3 urls

In [3]:
from urllib.parse import urlparse

bucket = "sds-n-cumulus-prod-nisar-products"

s3_urls = [f"s3://{bucket}/{'/'.join(urlparse(url).path.split('/')[2:])}" for url in hdf5_urls]
s3_urls

['s3://sds-n-cumulus-prod-nisar-products/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001.h5',
 's3://sds-n-cumulus-prod-nisar-products/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5',
 's3://sds-n-cumulus-prod-nisar-products/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001.h5',
 's3://sds-n-cumulus-prod-nisar-products/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T024619_20251228T024654_X05009_N_F_J_001/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T024619_20251228T024654_X05009_N_F_J_0

### 3d. Use an Earthdata Login Bearer Token to get S3 bucket access credentials

:::{figure} ../assets/edl_token.png
:alt: Image of the Earthdata Login Generate Token web page
:width: 75%
:align: left

View or generate a Bearer Token in "Generate Token" tab of the Profile page in your Earthdata Login account: https://urs.earthdata.nasa.gov/profile
:::

:::{note} Temporary S3 credentials

The S3 credentials acquired below expire after 1 hour.
:::

In [4]:
from getpass import getpass
import json
import s3fs
import urllib

token = getpass("Enter your EDL Bearer Token")
prefix = "NISAR_L2_GCOV_BETA_V1"

event = {
    "CredentialsEndpoint": "https://nisar.asf.earthdatacloud.nasa.gov/s3credentials",
    "BearerToken": token,
    "Bucket": bucket,
    "Prefix": prefix,
}

# Get temporary download credentials
tea_url = event["CredentialsEndpoint"]
bearer_token = event["BearerToken"]
req = urllib.request.Request(
    url=tea_url,
    headers={"Authorization": f"Bearer {bearer_token}"}
)
with urllib.request.urlopen(req) as f:
    creds = json.loads(f.read().decode())

fs = s3fs.S3FileSystem(
    key=creds["accessKeyId"],
    secret=creds["secretAccessKey"],
    token=creds["sessionToken"],
)

Enter your EDL Bearer Token ········


<hr>

(asf_search_stream-with)=
## 4. Load a single GCOV product using a Python `with` statement

 <br>

:::{note} This requires that you do all of your work on the dataset within the `with` block

Using `with` automatically closes an open file handle once out of scope of the `with` block. This is memory-safe but can also cause problems.

`with` statements are most appropriate when performing limited operations on a single data product. 

They are less useful when working with a time series, where you probably want to have multiple files open at once.
:::

### 4a. Open a GCOV file in a `with` block and compute the mean of an HHHH spatial subset

In [5]:
%%time
import xarray as xr
import rioxarray

s3_url = s3_urls[0]

kwargs = {
    "cache_type": "background",
    "block_size": 16 * 1024 * 1024,  # 16 MB
}

with fs.open(s3_url, "rb", **kwargs) as f:
    dt = xr.open_datatree(f, engine="h5netcdf", decode_timedelta=False, phony_dims="access")

    ### Perform any calculations and save any computed results for future access here, inside the `with` block ###
    frequencyA = dt["/science/LSAR/GCOV/grids/frequencyA"] # access the frequency A data
    projection = frequencyA.projection.attrs['epsg_code'].item() # access the GCOV product's projection
    hhhh = frequencyA.HHHH # access frequency A's HHHH band
    hhhh = hhhh.rio.write_crs(projection) # write the project to the HHHH data for easy lat/lon subsetting

    # subset the data
    subset_hhhh = hhhh.rio.clip_box(
        minx=40.8463, miny=13.2553,
        maxx=40.8574, maxy=13.2684, 
        crs="EPSG:4326"
    )

    # save the mean of HHHH subset for use outside of the `with` block
    subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
    print(f'subset_hhhh_mean: {subset_hhhh_mean}\n')

subset_hhhh_mean: 0.07500407099723816

CPU times: user 1.68 s, sys: 417 ms, total: 2.1 s
Wall time: 3.53 s


### 4b. View the `DataTree`

You can look at the `DataTree` outside the `with` block, but it only contains pointers to the data and the file is closed, so you can no longer access any of the data values to which it points

In [6]:
dt

<xarray.DataTree>
Group: /
│   Attributes:
│       Conventions:         CF-1.7
│       contact:             nisar-sds-ops@jpl.nasa.gov
│       institution:         NASA JPL
│       mission_name:        NISAR
│       reference_document:  D-102274 NISAR NASA SDS Product Specification Level-...
│       title:               NISAR L2 GCOV Product
└── Group: /science
    └── Group: /science/LSAR
        ├── Group: /science/LSAR/GCOV
        │   ├── Group: /science/LSAR/GCOV/grids
        │   │   ├── Group: /science/LSAR/GCOV/grids/frequencyA
        │   │   │       Dimensions:                (yCoordinates: 16704, xCoordinates: 17064,
        │   │   │                                   phony_dim_0: 2)
        │   │   │       Coordinates:
        │   │   │         * yCoordinates           (yCoordinates) float64 134kB 1.588e+06 ... 1.254e+06
        │   │   │         * xCoordinates           (xCoordinates) float64 137kB 6.041e+05 ... 9.454e+05
        │   │   │       Dimensions without coordinates: phony_dim_0
        │   │   │       Data variables:
        │   │   │           HHHH                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           HVHV                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           listOfCovarianceTerms  (phony_dim_0) |S4 8B ...
        │   │   │           listOfPolarizations    (phony_dim_0) |S2 4B ...
        │   │   │           mask                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfLooks          (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfSubSwaths      uint8 1B ...
        │   │   │           projection             uint32 4B ...
        │   │   │           rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           xCoordinateSpacing     float64 8B ...
        │   │   │           yCoordinateSpacing     float64 8B ...
        │   │   └── Group: /science/LSAR/GCOV/grids/frequencyB
        │   │           Dimensions:                (yCoordinates: 4176, xCoordinates: 4266,
        │   │                                       phony_dim_1: 2)
        │   │           Coordinates:
        │   │             * yCoordinates           (yCoordinates) float64 33kB 1.588e+06 ... 1.254e+06
        │   │             * xCoordinates           (xCoordinates) float64 34kB 6.041e+05 ... 9.453e+05
        │   │           Dimensions without coordinates: phony_dim_1
        │   │           Data variables:
        │   │               HHHH                   (yCoordinates, xCoordinates) float32 71MB ...
        │   │               HVHV                   (yCoordinates, xCoordinates) float32 71MB ...
        │   │               listOfCovarianceTerms  (phony_dim_1) |S4 8B ...
        │   │               listOfPolarizations    (phony_dim_1) |S2 4B ...
        │   │               mask                   (yCoordinates, xCoordinates) float32 71MB ...
        │   │               numberOfLooks          (yCoordinates, xCoordinates) float32 71MB ...
        │   │               numberOfSubSwaths      uint8 1B ...
        │   │               projection             uint32 4B ...
        │   │               rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 71MB ...
        │   │               xCoordinateSpacing     float64 8B ...
        │   │               yCoordinateSpacing     float64 8B ...
        │   └── Group: /science/LSAR/GCOV/metadata
        │       ├── Group: /science/LSAR/GCOV/metadata/attitude
        │       │       Dimensions:       (phony_dim_2: 292, phony_dim_3: 3, phony_dim_4: 4)
        │       │       Dimensions without coordinates: phony_dim_2, phony_dim_3, phony_dim_4
        │       │       Data variables:
        │       │           attitudeType  |S10 10B ...
        │       │           eulerAngles   (phony_dim_2, phony_dim_3) float64 7kB ...
        │       │           quaternions   (phony_dim_2, phony_dim_4) float64 9kB ...
        │  

### 4c. Try (and fail) to compute a value in the `DataTree`

The file was closed upon exiting the `with` block so trying to compute a value from the `DataTree` will fail with a `ValueError: I/O operation on closed file.`

In [7]:
dt.compute()

ValueError: I/O operation on closed file.

#### However, `subset_hhhh_mean` was computed and saved to memory inside the `with` block, so we can still see its value

In [8]:
subset_hhhh_mean

0.07500407099723816

<hr>

(asf_search_stream-open_many)=
## 5. Load multiple GCOV Products at once

:::{note} Lets speed things up a bit

If you are not working with all of the groups in an HDF5 file, there is no need to spend time loading the entire file over the network, as was done in the example above.

For the remaining examples in this notebook, we will access only the Frequency A data, stored in the group `/science/LSAR/GCOV/grids/frequencyA`.
:::

### 5a. Iterate through a list of HDF5 S3 bucket URLs, and open the `/science/LSAR/GCOV/grids/frequencyA` for each

This leaves the files open for later use, which means you should manually close them when finished in order to prevent memory leaks.

In [9]:
%%time
import xarray as xr
import rioxarray

# Explore the DataTree rendering above in Step 4 for a complete list of available groups 
group_path = "/science/LSAR/GCOV/grids/frequencyA" # change this to any GCOV HDF5 group you wish

kwargs = {
    "cache_type": "background",
    "block_size": 16 * 1024 * 1024,  # 16 MB
}

files = [fs.open(url, "rb", **kwargs) for url in s3_urls]

datatrees = [
    xr.open_datatree(
        f,
        engine="h5netcdf",
        decode_timedelta=False,
        phony_dims="access",
        chunks="auto",
        group=group_path,
    )
    for f in files
]

CPU times: user 787 ms, sys: 407 ms, total: 1.19 s
Wall time: 4.52 s


### 5b. Since the files remain open, we can access still access them and perform computations on their contents

Iterate through the `DataTrees`, calculating subset HHHH mean values

In [10]:
for tree in datatrees:
    projection = tree.projection.attrs['epsg_code'].item()
    hhhh = tree.HHHH
    hhhh = hhhh.rio.write_crs(projection)
    
    subset_hhhh = hhhh.rio.clip_box(
        minx=40.8463, miny=13.2553,
        maxx=40.8574, maxy=13.2684, 
        crs="EPSG:4326"
    )
    subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
    print(subset_hhhh_mean)

0.07500407099723816
0.07547999173402786
0.0776161402463913
0.07449550926685333
0.07419200986623764


<hr>

(asf_search_stream-ts)=
## 6. Stream a time series

Use the multiple files we opened in Step 5 to create a GCOV time series

### 6a. Define a function to extract datetimes for a `time` dimension

In [11]:
import re
from urllib.parse import urlparse
from datetime import datetime
from pathlib import PurePosixPath

NISAR_TS_RE = re.compile(r"_(\d{8}T\d{6})_")

def nisar_start_time_from_url(s3_url: str) -> datetime:
    path = urlparse(s3_url).path
    name = PurePosixPath(path).name
    
    m = NISAR_TS_RE.search(name)
    if not m:
        raise ValueError(f"No NISAR timestamp found in: {s3_url}")
    
    return datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")

### 6b. Create a list of datetimes for a `time` dimension

In [12]:
dts = [nisar_start_time_from_url(url) for url in s3_urls]
dts

[datetime.datetime(2025, 11, 22, 2, 46, 18),
 datetime.datetime(2025, 12, 4, 2, 46, 18),
 datetime.datetime(2025, 12, 16, 2, 46, 19),
 datetime.datetime(2025, 12, 28, 2, 46, 19),
 datetime.datetime(2026, 1, 9, 2, 46, 20)]

### 6c. Create an `xarray.DataArray` containing a `time` dimension for each open GCOV group

In [13]:
dataarrays = [
    tree.ds.assign_coords(time=dt).expand_dims(time=1)
    for dt, tree in zip(dts, datatrees)
]

dataarrays[0].time

<xarray.DataArray 'time' (time: 1)> Size: 8B
array(['2025-11-22T02:46:18.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 8B 2025-11-22T02:46:18

In [14]:
for da in dataarrays:
    print(da.dims)

FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})


### 6d. Concatenate the `xarray.DataArray`s into a single `xarray.Dataset` time series 

In [15]:
ts = xr.concat(dataarrays, dim="time")
ts

<xarray.Dataset> Size: 29GB
Dimensions:                (time: 5, yCoordinates: 16704, xCoordinates: 17064,
                            phony_dim_0: 2)
Coordinates:
  * time                   (time) datetime64[ns] 40B 2025-11-22T02:46:18 ... ...
  * yCoordinates           (yCoordinates) float64 134kB 1.588e+06 ... 1.254e+06
  * xCoordinates           (xCoordinates) float64 137kB 6.041e+05 ... 9.454e+05
Dimensions without coordinates: phony_dim_0
Data variables:
    HHHH                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    HVHV                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    listOfCovarianceTerms  (time, phony_dim_0) |S4 40B dask.array<chunksize=(1, 2), meta=np.ndarray>
    listOfPolarizations    (time, phony_dim_0) |S2 20B dask.array<chunksize=(1, 2), meta=np.ndarray>
    mask                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    numberOfLooks          (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    numberOfSubSwaths      (time) uint8 5B 1 1 1 1 1
    projection             (time) uint32 20B 32637 32637 32637 32637 32637
    rtcGammaToSigmaFactor  (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    xCoordinateSpacing     (time) float64 40B 20.0 20.0 20.0 20.0 20.0
    yCoordinateSpacing     (time) float64 40B -20.0 -20.0 -20.0 -20.0 -20.0

<hr>

(asf_search_stream-close)=
## 7. Close open files

Close all open files to prevent memory leaks.

### 7a. Close the files

In [16]:
for f in files:
    f.close()

#### 7b. With closed files, we can no longer compute values in the `xarray` data structures that we have created

The cell below will return a `ValueError: I/O operation on closed file.`

In [17]:
ts.compute()

ValueError: I/O operation on closed file.

<hr>

(asf_search_stream-summary)=
## 8. Summary
You now have the tools and knowledge that you need to search with `asf_search`, generate temporary S3 bucket credentials from an Earthdata Login Bearer Token, stream data from S3 with `s3fs`, and load them into `xarray` data structures.

<hr>

(asf_search_stream-resources)=
## 9. Resources and references
 - [asf_search](https://github.com/asfadmin/Discovery-asf_search)
 - [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/aws-s3-access/)
 
**Author:** Alex Lewandowski